In [ ]:
from ingest import load_faq_data, build_index
from rag_helper import RAGBase
from openai import OpenAI
import json

documents = load_faq_data()
index = build_index(documents)
llm_client = OpenAI()

messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

response = llm_client.responses.create(
        model = "gpt-5.4-mini",
        input = messages,
        tools = [search_tool]
    )

print(response.output[0])

call = response.output[0]

args = json.loads(call.arguments)

results = search(**args)
json_text = json.dumps(results, indent=2)

messages.extend(response.output)

messages.append({
    "type": "function_call_output",
    "call_id": call.call_id,
    "output":json_text
})

response = llm_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output_text



ResponseFunctionToolCall(arguments='{"query":"Can I join the course if I just discovered it? enrollment join late registration FAQ"}', call_id='call_8znJ6oqnvgKNOoJIv2hz1Akk', name='search', type='function_call', id='fc_06934a3acbf36509006af77d9ca2c087d184b8093462649a4b', async_=None, caller=None, namespace=None, status='completed')


'Yes — you can still join.\n\nIf you want a certificate, though, you need to submit your project while submissions are still open.'